# Train Monte Carlo Control

Monte Carlo Control averages complete episode returns for epsilon-greedy control. The experiment uses deterministic `FrozenLake-v1` so the learned table can be inspected directly.

## Defining update

$$Q(S_t,A_t)\leftarrow Q(S_t,A_t)+\frac{G_t-Q(S_t,A_t)}{N(S_t,A_t)}.$$

Here $G_t$ is the complete sampled return and $N(S_t,A_t)$ the visit count for the state-action pair.

In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np

from aprenderl import MonteCarloControl, MonteCarloControlConfig
from aprenderl.utils import evaluate_policy

ENV_ID = "FrozenLake-v1"

In [ ]:
env = gym.make(ENV_ID, is_slippery=False)
config = MonteCarloControlConfig(
    first_visit=True,
    exploration_steps=3_000,
    seed=7,
)

agent = MonteCarloControl(env, config=config)
agent.learn(total_timesteps=5_000, progress_bar=False)
env.close()

In [ ]:
print("Q-table shape:", agent.q_table.shape)
print("Greedy policy:")
print(agent.q_table.argmax(axis=1).reshape(4, 4))
print("Updates:", agent.num_updates)

In [ ]:
returns = np.asarray(agent.episode_returns)
window = min(20, len(returns))
moving_average = np.convolve(
    returns, np.ones(window) / window, mode="valid"
)

plt.figure(figsize=(8, 4))
plt.plot(returns, alpha=0.35, label="Episode return")
plt.plot(
    np.arange(window - 1, len(returns)),
    moving_average,
    label=f"{window}-episode average",
)
plt.xlabel("Episode")
plt.ylabel("Return")
plt.title(f"Monte Carlo Control training on {ENV_ID}")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## Watch the trained policy

This opens a window and runs 5 episodes using deterministic actions.

In [ ]:

evaluation_env = gym.make(
    ENV_ID, render_mode="human", is_slippery=False
)
try:
    result = evaluate_policy(
        agent, evaluation_env, episodes=5, deterministic=True
    )
finally:
    evaluation_env.close()

print("Episode returns:", result.returns)
print(f"Mean return: {result.mean_return:.1f} +/- {result.return_std:.1f}")